# Requêtes sources du One Pager

Une section par KPI (numérotation de `docs/notes_et_questions.md`).

- **Requête retenue** : détail au grain fin (une ligne par ligne de production, par batch...), c'est elle qui alimentera les notebooks puis Power BI.
- **Contrôle** : donne la valeur du KPI à date du jour, uniquement pour vérifier la requête retenue.

Les KPI de déploiement D2, D3 et D4 viennent uniquement du référentiel des activations de l'équipe DS (une ligne par ligne de production et par module). Nom de table provisoire : `mal_maite_common_prd.gold.ref_production_line_deployment`.

| KPI | Statut |
|---|---|
| D1 – Data digitalization | ✅ validé (14) |
| D2 – Metamodels | ⏳ en attente de la table DS |
| D3 – AI Recommendations | ⏳ en attente de la table DS |
| D4 – Time Tracking & Production Notes | ⏳ en attente de la table DS |
| Autres KPI | requêtes initiales, à reprendre une par une |

## D1 – Data digitalization ✅
Lignes ayant plus de 180 batchs non supprimés (tous statuts, cf. PO1). Date = `created_at` du 181e batch.

In [0]:
%sql
WITH batches_numerotes AS (
    SELECT
        pl.id_plant_production_line AS production_line_id,
        pl.common_code              AS production_line_code,
        pl.name                     AS production_line_name,
        b.id_batch,
        b.created_at,
        ROW_NUMBER() OVER (
            PARTITION BY pl.id_plant_production_line
            ORDER BY b.created_at, b.id_batch
        ) AS rang_batch
    FROM ext_mal_psql_maite_vision_board_prd.public.batches b
    INNER JOIN ext_mal_psql_maite_vision_board_prd.public.plants_production_lines pl
        ON pl.id_plant_production_line = b.production_line
    WHERE b.deleted = false
)
SELECT
    production_line_id,
    production_line_code,
    production_line_name,
    to_date(created_at) AS digitalization_date
FROM batches_numerotes
WHERE rang_batch = 181
ORDER BY digitalization_date;

In [0]:
%sql
SELECT COUNT(*) AS kpi_data_digitalization
FROM (
    SELECT pl.id_plant_production_line
    FROM ext_mal_psql_maite_vision_board_prd.public.batches b
    INNER JOIN ext_mal_psql_maite_vision_board_prd.public.plants_production_lines pl
        ON pl.id_plant_production_line = b.production_line
    WHERE b.deleted = false
    GROUP BY pl.id_plant_production_line
    HAVING COUNT(b.id_batch) > 180
) lignes_digitalisees;

## D2 – Metamodels ⏳
Lignes actives sur le module `metamodel` du référentiel DS. En attente de la livraison de la table DS.

In [0]:
%sql
SELECT
    production_line_id,
    production_line_code,
    site,
    is_active,
    valid_from,
    valid_to
FROM mal_maite_common_prd.gold.ref_production_line_deployment
WHERE module = 'metamodel'
ORDER BY valid_from;

In [0]:
%sql
SELECT COUNT(DISTINCT production_line_id) AS kpi_metamodels
FROM mal_maite_common_prd.gold.ref_production_line_deployment
WHERE module = 'metamodel'
  AND is_active = true;

## D3 – AI Recommendations ⏳
Lignes actives sur le module `recommendations` du référentiel DS. En attente de la livraison de la table DS.

In [0]:
%sql
SELECT
    production_line_id,
    production_line_code,
    site,
    is_active,
    valid_from,
    valid_to
FROM mal_maite_common_prd.gold.ref_production_line_deployment
WHERE module = 'recommendations'
ORDER BY valid_from;

In [0]:
%sql
SELECT COUNT(DISTINCT production_line_id) AS kpi_ai_recommendations
FROM mal_maite_common_prd.gold.ref_production_line_deployment
WHERE module = 'recommendations'
  AND is_active = true;

## D4 – Time Tracking & Production Notes ⏳
Lignes actives à la fois sur `process_time` (TRS) et sur `production_notes` du référentiel DS. En attente de la livraison de la table DS.

In [0]:
%sql
SELECT
    pt.production_line_id,
    pt.production_line_code,
    pt.site,
    (pt.is_active AND pn.is_active)          AS is_active,
    GREATEST(pt.valid_from, pn.valid_from)   AS valid_from,   -- les deux modules actifs
    NULLIF(
        LEAST(
            COALESCE(pt.valid_to, DATE'9999-12-31'),
            COALESCE(pn.valid_to, DATE'9999-12-31')
        ),
        DATE'9999-12-31'
    )                                        AS valid_to      -- le premier désactivé
FROM mal_maite_common_prd.gold.ref_production_line_deployment pt
INNER JOIN mal_maite_common_prd.gold.ref_production_line_deployment pn
    ON  pn.production_line_id = pt.production_line_id
    AND pn.module = 'production_notes'
WHERE pt.module = 'process_time'
ORDER BY valid_from;

In [0]:
%sql
SELECT COUNT(DISTINCT pt.production_line_id) AS kpi_time_tracking_production_notes
FROM mal_maite_common_prd.gold.ref_production_line_deployment pt
INNER JOIN mal_maite_common_prd.gold.ref_production_line_deployment pn
    ON  pn.production_line_id = pt.production_line_id
    AND pn.module = 'production_notes'
WHERE pt.module = 'process_time'
  AND pt.is_active = true
  AND pn.is_active = true;

## Requêtes initiales des autres KPI
A reprendre KPI par KPI (même méthode : requête retenue + contrôle).

In [0]:
%sql
------Maltimize unique users
SELECT * FROM ext_mal_psql_maite_vision_board_prd.public.profiles_users
WHERE deleted = 'false';

### U2 – Dashboards PowerBI unique users (requête initiale)
Pas encore de requête : source non disponible.

In [0]:
%sql
------Uncaptured Aveva data
SELECT COUNT(flg_echec) 
FROM `mal_maite_bi_prd`.`recommendations_monitoring`.`fact_monitoring_auto_manual_entries`
WHERE succes_echec = 'ECHEC'
      AND flg_auto = 1
      AND Id_alerting = 1;

In [0]:
%sql
------Missing manual entries
SELECT SUM(flg_echec)
FROM `mal_maite_bi_prd`.`recommendations_monitoring`.`fact_monitoring_auto_manual_entries`
WHERE flg_echec = 1 
AND has_manual_value = 1 
AND has_auto_value = 0;

In [0]:
%sql
------Missing auto entries
SELECT SUM(flg_echec)
FROM `mal_maite_bi_prd`.`recommendations_monitoring`.`fact_monitoring_auto_manual_entries`
WHERE flg_echec = 1 
AND has_manual_value = 0 
AND has_auto_value = 1;

In [0]:
%sql
------Missing auto & manual entries
SELECT SUM(flg_echec)
FROM `mal_maite_bi_prd`.`recommendations_monitoring`.`fact_monitoring_auto_manual_entries`
WHERE flg_echec = 1 
AND has_manual_value = 1
AND has_auto_value = 1;

In [0]:
%sql
------Total recommandations echec ou succes
SELECT 
    SUM(
        COALESCE(flg_echec, 0) 
        + 
        COALESCE(flg_succes, 0)
    ) AS reco_total
FROM `mal_maite_bi_prd`.`recommendations_monitoring`.`fact_monitoring_auto_manual_entries`;